<a href="https://colab.research.google.com/github/IA-BSA/SIA-VERDE/blob/main/TFM_SIAVERDE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'https://raw.githubusercontent.com/IA-BSA/SIA-VERDE/refs/heads/main/calidad_aire.csv'

In [ ]:
'https://raw.githubusercontent.com/IA-BSA/SIA-VERDE/refs/heads/main/trafico.csv'

In [ ]:
import pandas as pd

calidad_aire_url = 'https://raw.githubusercontent.com/IA-BSA/SIA-VERDE/refs/heads/main/calidad_aire.csv'
trafico_url = 'https://raw.githubusercontent.com/IA-BSA/SIA-VERDE/refs/heads/main/trafico.csv'

df_calidad_aire = pd.read_csv(calidad_aire_url)
df_trafico = pd.read_csv(trafico_url, sep=';')

print("Primeras 5 filas de la tabla 'calidad_aire':")
display(df_calidad_aire.head())

print("\nPrimeras 5 filas de la tabla 'trafico':")
display(df_trafico.head())

In [ ]:
# Get the name of the current single column
current_trafico_column_name = df_trafico.columns[0]

# Split the single column by comma as requested by the user
# This will likely result in a single column if no commas are present in the data
split_trafico_data = df_trafico[current_trafico_column_name].str.split(',', expand=True)

# Assign the new columns back to df_trafico
df_trafico = split_trafico_data

# Display the first 5 rows to show the result
print("Primeras 5 filas de la tabla 'trafico' después de intentar separar por coma:")
display(df_trafico.head())

In [ ]:
# Convert 'FECHA' in df_calidad_aire to datetime objects
df_calidad_aire['FECHA'] = pd.to_datetime(df_calidad_aire['FECHA'], format='%d-%m-%Y')

# Convert 'fecha' in df_trafico to datetime objects
df_trafico['fecha'] = pd.to_datetime(df_trafico['fecha'], format='%d/%m/%Y %H:%M')

# Rename 'fecha' column in df_trafico to 'FECHA'
df_trafico = df_trafico.rename(columns={'fecha': 'FECHA'})

print("Primeras 5 filas de df_calidad_aire con el formato de fecha unificado:")
display(df_calidad_aire.head())

print("\nPrimeras 5 filas de df_trafico con el formato de fecha y nombre de columna unificados:")
display(df_trafico.head())

print("\nTipos de datos de df_calidad_aire después de la unificación:")
df_calidad_aire.info()

print("\nTipos de datos de df_trafico después de la unificación:")
df_trafico.info()

In [ ]:
# Aggregate df_trafico to daily intervals with specified aggregations
# Extract the date part from the 'FECHA' column for daily aggregation
df_trafico_diario = df_trafico.groupby(['id', df_trafico['FECHA'].dt.date]).agg(
    intensidad_diaria=('intensidad', 'sum'),
    ocupacion_diaria=('ocupacion', 'mean'),
    carga_diaria=('carga', 'mean')
).reset_index()

# Rename the date column to 'FECHA' to maintain consistency
df_trafico_diario = df_trafico_diario.rename(columns={'FECHA': 'FECHA_DIARIA'})

# Convert 'FECHA_DIARIA' back to datetime objects for consistency if needed later
df_trafico_diario['FECHA_DIARIA'] = pd.to_datetime(df_trafico_diario['FECHA_DIARIA'])

print("Primeras 5 filas de la tabla 'trafico' con intervalo diario y agregaciones actualizadas:")
display(df_trafico_diario.head())

print("\nTipos de datos de df_trafico_diario:")
df_trafico_diario.info()

In [ ]:
# Rename 'FECHA_DIARIA' to 'FECHA' in df_trafico_diario for a consistent merge key
df_trafico_diario = df_trafico_diario.rename(columns={'FECHA_DIARIA': 'FECHA'})

# Merge df_calidad_aire and df_trafico_diario on the 'FECHA' column
# An inner merge is chosen to only include dates present in both dataframes
df_merged = pd.merge(df_calidad_aire, df_trafico_diario, on='FECHA', how='inner')

print("Primeras 5 filas de la tabla combinada:")
display(df_merged.head())

print("\nTipos de datos de la tabla combinada:")
df_merged.info()

In [ ]:
print(f"La tabla combinada 'df_merged' tiene {len(df_merged)} filas.")

In [ ]:
print(f"Número de fechas únicas en df_calidad_aire: {df_calidad_aire['FECHA'].nunique()}")
print(f"Número de fechas únicas en df_trafico_diario: {df_trafico_diario['FECHA'].nunique()}")

# Calculate the number of unique 'id's per date in df_trafico_diario
ids_per_date = df_trafico_diario.groupby('FECHA')['id'].nunique()

print("\nDistribución del número de IDs únicos por fecha en df_trafico_diario:")
display(ids_per_date.value_counts())

print(f"\nPromedio de IDs únicos por fecha en df_trafico_diario: {ids_per_date.mean():.2f}")

# Let's also check the number of unique dates in the merged dataframe
print(f"\nNúmero de fechas únicas en df_merged: {df_merged['FECHA'].nunique()}")

Como puedes ver, `df_trafico_diario` tiene múltiples `id`s asociados a cada fecha. Por ejemplo, si en un día tienes 4 `id`s diferentes en `df_trafico_diario` y ese mismo día existe en `df_calidad_aire`, la fila de `df_calidad_aire` para ese día se duplicará 4 veces en la tabla fusionada, una por cada `id` de tráfico. Por lo tanto, el número de filas en `df_merged` es el resultado de la combinación de todas las fechas coincidentes y todos los `id`s de tráfico para esas fechas.

In [ ]:
# Aggregating df_trafico_diario further to have only one row per date
# We'll sum 'intensidad_diaria' and average 'ocupacion_diaria' and 'carga_diaria' across all 'id's for each date
df_trafico_diario_unificado = df_trafico_diario.groupby('FECHA').agg(
    intensidad_total_diaria=('intensidad_diaria', 'sum'),
    ocupacion_media_diaria=('ocupacion_diaria', 'mean'),
    carga_media_diaria=('carga_diaria', 'mean')
).reset_index()

print("Primeras 5 filas de la tabla 'trafico' unificada por fecha:")
display(df_trafico_diario_unificado.head())

print("\nTipos de datos de df_trafico_diario_unificado:")
df_trafico_diario_unificado.info()

In [ ]:
# Now, merge df_calidad_aire with the newly aggregated df_trafico_diario_unificado
# This merge will avoid the row duplication as df_trafico_diario_unificado has only one entry per date
df_merged_re_aggregated = pd.merge(df_calidad_aire, df_trafico_diario_unificado, on='FECHA', how='inner')

print("Primeras 5 filas de la tabla combinada re-agregada:")
display(df_merged_re_aggregated.head())

print("\nTipos de datos de la tabla combinada re-agregada:")
df_merged_re_aggregated.info()

print(f"\nLa tabla combinada re-agregada 'df_merged_re_aggregated' tiene {len(df_merged_re_aggregated)} filas.")

### Filtrar y reagregar datos de tráfico para sensores específicos (3950 y 3951)

El usuario ha señalado que hay datos duplicados en 2024 y 2025 y desea mantener solo los datos de los sensores (ID) 3950 y 3951 en la tabla de tráfico.

Para lograr esto, primero filtraremos `df_trafico_diario` por los `id`s deseados. Luego, reagregaremos las métricas de tráfico por `FECHA` (sumando `intensidad_diaria` y promediando `ocupacion_diaria` y `carga_diaria`) para estos sensores específicos. Finalmente, fusionaremos esta tabla reagregada con `df_calidad_aire`.

In [ ]:
# Define los IDs de los sensores que quieres mantener
sensores_interes = [3950, 3951]

# Filtra df_trafico_diario para incluir solo los sensores de interés
df_trafico_filtrado = df_trafico_diario[df_trafico_diario['id'].isin(sensores_interes)]

# Agrega df_trafico_filtrado por FECHA, sumando intensidad y promediando ocupacion y carga
df_trafico_reagregado_filtrado = df_trafico_filtrado.groupby('FECHA').agg(
    intensidad_total_diaria=('intensidad_diaria', 'sum'),
    ocupacion_media_diaria=('ocupacion_diaria', 'mean'),
    carga_media_diaria=('carga_diaria', 'mean')
).reset_index()

print("Primeras 5 filas de la tabla de tráfico reagregada y filtrada por sensores:")
display(df_trafico_reagregado_filtrado.head())

print("\nTipos de datos de df_trafico_reagregado_filtrado:")
df_trafico_reagregado_filtrado.info()

print(f"\nNúmero de filas en df_trafico_reagregado_filtrado: {len(df_trafico_reagregado_filtrado)}")

### Fusión final de `df_calidad_aire` y los datos de tráfico filtrados y reagregados

Ahora, fusionaremos `df_calidad_aire` con `df_trafico_reagregado_filtrado` utilizando la columna `FECHA` como clave. Esto debería proporcionar la tabla final deseada sin duplicaciones debidas a IDs de sensores no deseados.

In [ ]:
# Fusiona df_calidad_aire con el nuevo df_trafico_reagregado_filtrado
df_merged_final = pd.merge(df_calidad_aire, df_trafico_reagregado_filtrado, on='FECHA', how='inner')

print("Primeras 5 filas de la tabla combinada final (con sensores 3950 y 3951 solamente):")
display(df_merged_final.head())

print("\nTipos de datos de la tabla combinada final:")
df_merged_final.info()

print(f"\nLa tabla combinada final 'df_merged_final' tiene {len(df_merged_final)} filas.")

### Verificación de valores nulos en la tabla combinada final

Para asegurarnos de que la tabla combinada final esté limpia, realizaremos una verificación rápida de valores nulos.

In [ ]:
null_values_df_final = df_merged_final[df_merged_final.isnull().any(axis=1)]

if not null_values_df_final.empty:
    print("Filas con valores nulos en df_merged_final:")
    display(null_values_df_final)
else:
    print("No hay valores nulos en el DataFrame df_merged_final.")

In [ ]:
print("Primeras 5 filas de df_merged_final:")
display(df_merged_final.head())

print("\nÚltimas 5 filas de df_merged_final:")
display(df_merged_final.tail())

In [ ]:
# Guardar df_merged_final en un archivo CSV
df_merged_final.to_csv('df_merged_final.csv', index=False)
print("DataFrame 'df_merged_final' guardado exitosamente como 'df_merged_final.csv'")

### Agregando Características de Estacionalidad

Vamos a extraer el mes y el día de la semana de la columna `FECHA` para capturar posibles patrones estacionales que podrían influir en la calidad del aire.

In [ ]:
# Extraer el mes de la columna FECHA
df_merged_final['mes'] = df_merged_final['FECHA'].dt.month

# Extraer el día de la semana de la columna FECHA (0 = Lunes, 6 = Domingo)
df_merged_final['dia_semana'] = df_merged_final['FECHA'].dt.dayofweek

print("Primeras 5 filas de df_merged_final con nuevas características de estacionalidad:")
display(df_merged_final.head())

### Agregando la Contaminación del Día Anterior como Característica

Vamos a crear una nueva columna que contenga el `VALOR_DIARIO` del día anterior. Esta característica de rezago puede ayudar a los modelos a entender la dependencia temporal de la calidad del aire.

In [ ]:
# Asegurarse de que el DataFrame esté ordenado por FECHA para calcular correctamente el rezago
df_merged_final = df_merged_final.sort_values(by='FECHA')

# Crear la columna de 'VALOR_DIARIO_DIA_ANTERIOR'
df_merged_final['VALOR_DIARIO_DIA_ANTERIOR'] = df_merged_final.groupby(['PROVINCIA', 'MUNICIPIO', 'ESTACION', 'MAGNITUD'])['VALOR_DIARIO'].shift(1)

# Manejar los valores NaN resultantes de la operación shift (primer día de cada grupo)
# Podríamos rellenarlos con la media, mediana, o simplemente dejarlos y el modelo los manejará (o los eliminaremos para el entrenamiento).
# Por ahora, los dejaremos como NaN, y el train_test_split los eliminará si hay filas con NaN.

print("Primeras 10 filas de df_merged_final con la nueva característica de rezago:")
display(df_merged_final.head(10))

print("Número de valores nulos en 'VALOR_DIARIO_DIA_ANTERIOR':")
print(df_merged_final['VALOR_DIARIO_DIA_ANTERIOR'].isnull().sum())

### Preparación de Datos para Modelos de Regresión

Primero, definiremos nuestras variables predictoras (features) y la variable objetivo (target). Luego, dividiremos los datos en conjuntos de entrenamiento y prueba para evaluar el rendimiento de los modelos.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Definir las características (X) y la variable objetivo (y)
# Suponemos que queremos predecir VALOR_DIARIO de calidad del aire basándonos en métricas de tráfico y estacionalidad.
X = df_merged_final[['intensidad_total_diaria', 'ocupacion_media_diaria', 'carga_media_diaria', 'mes', 'dia_semana', 'VALOR_DIARIO_DIA_ANTERIOR']]
y = df_merged_final['VALOR_DIARIO']

# Como 'VALOR_DIARIO_DIA_ANTERIOR' tiene un NaN, necesitamos manejarlo.
# La forma más simple para la división es eliminar las filas con NaN en X e y.
df_clean = df_merged_final.dropna(subset=X.columns.tolist() + [y.name])
X_clean = df_clean[X.columns]
y_clean = df_clean[y.name]

# Dividir los datos en conjuntos de entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

print(f"Tamaño del conjunto de entrenamiento: {len(X_train)} filas")
print(f"Tamaño del conjunto de prueba: {len(X_test)} filas")

### 1. Modelo de Regresión Lineal

Entrenaremos un modelo de regresión lineal y evaluaremos su rendimiento utilizando las métricas R2, MAE y RMSE.

In [ ]:
# Inicializar y entrenar el modelo de Regresión Lineal
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred_linear = linear_model.predict(X_test)

# Calcular métricas de evaluación
r2_linear = r2_score(y_test, y_pred_linear)
mae_linear = mean_absolute_error(y_test, y_pred_linear)
rmse_linear = np.sqrt(mean_squared_error(y_test, y_pred_linear))

print("--- Regresión Lineal ---")
print(f"R2 Score: {r2_linear:.4f}")
print(f"Mean Absolute Error (MAE): {mae_linear:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_linear:.4f}")

### 2. Modelo de Random Forest

Ahora, entrenaremos un modelo de Random Forest Regressor y evaluaremos su rendimiento.

In [ ]:
# Inicializar y entrenar el modelo de Random Forest Regressor
# Se usa random_state para reproducibilidad
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred_rf = rf_model.predict(X_test)

# Calcular métricas de evaluación
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("--- Random Forest Regressor ---")
print(f"R2 Score: {r2_rf:.4f}")
print(f"Mean Absolute Error (MAE): {mae_rf:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_rf:.4f}")

### Comparación Visual de Predicciones

Visualizaremos las predicciones de ambos modelos frente a los valores reales para tener una idea gráfica de su rendimiento.

In [ ]:
plt.figure(figsize=(14, 6))

# Subplot para Regresión Lineal
plt.subplot(1, 2, 1) # 1 fila, 2 columnas, primer gráfico
sns.scatterplot(x=y_test, y=y_pred_linear, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Regresión Lineal: Predicciones vs Reales')
plt.xlabel('Valores Reales de VALOR_DIARIO')
plt.ylabel('Predicciones de VALOR_DIARIO')
plt.grid(True)

# Subplot para Random Forest Regressor
plt.subplot(1, 2, 2) # 1 fila, 2 columnas, segundo gráfico
sns.scatterplot(x=y_test, y=y_pred_rf, alpha=0.6, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Random Forest: Predicciones vs Reales')
plt.xlabel('Valores Reales de VALOR_DIARIO')
plt.ylabel('Predicciones de VALOR_DIARIO')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
meteo_url_1 = 'https://datos.madrid.es/dataset/300351-0-meteorologicos-diarios/resource/300351-6-meteorologicos-diarios-csv/download/300351-6-meteorologicos-diarios-csv.csv'
meteo_url_2 = 'https://datos.madrid.es/dataset/300351-0-meteorologicos-diarios/resource/300351-0-meteorologicos-diarios-csv/download/300351-0-meteorologicos-diarios-csv.csv'
meteo_url_3 = 'https://datos.madrid.es/dataset/300351-0-meteorologicos-diarios/resource/300351-1-meteorologicos-diarios-csv/download/300351-1-meteorologicos-diarios-csv.csv'

df_meteo_1 = pd.read_csv(meteo_url_1, sep=';')
df_meteo_2 = pd.read_csv(meteo_url_2, sep=';')
df_meteo_3 = pd.read_csv(meteo_url_3, sep=';')

### Primeras 5 filas de `df_meteo_1`

In [ ]:
display(df_meteo_1.head())

### Primeras 5 filas de `df_meteo_2`

In [ ]:
display(df_meteo_2.head())

### Primeras 5 filas de `df_meteo_3`

In [ ]:
display(df_meteo_3.head())

In [ ]:
# 1. Concatenar todas las tablas meteorológicas
df_meteo_combined = pd.concat([df_meteo_1, df_meteo_2, df_meteo_3])

print("Primeras 5 filas del DataFrame meteorológico combinado:")
display(df_meteo_combined.head())
print(f"Número de filas en df_meteo_combined: {len(df_meteo_combined)}")

### 2. y 3. Transformar y crear la columna `FECHA`

In [ ]:
# Identificar columnas de valores (DXX) y validación (VXX)
value_cols = [col for col in df_meteo_combined.columns if col.startswith('D')]
validation_cols = [col for col in df_meteo_combined.columns if col.startswith('V')]

# Deshacer la fusión de columnas 'D' y 'V' por separado
df_meteo_melted_values = df_meteo_combined.melt(
    id_vars=['PROVINCIA', 'MUNICIPIO', 'ESTACION', 'MAGNITUD', 'ANO', 'MES'],
    value_vars=value_cols,
    var_name='Day_raw',
    value_name='VALOR_METEO'
)
df_meteo_melted_validation = df_meteo_combined.melt(
    id_vars=['PROVINCIA', 'MUNICIPIO', 'ESTACION', 'MAGNITUD', 'ANO', 'MES'],
    value_vars=validation_cols,
    var_name='Validation_raw',
    value_name='VALIDACION_METEO'
)

# Extraer el número del día y convertir a entero
df_meteo_melted_values['Day'] = df_meteo_melted_values['Day_raw'].str[1:].astype(int)
df_meteo_melted_validation['Day'] = df_meteo_melted_validation['Validation_raw'].str[1:].astype(int)

# Unir los valores y la validación basándose en las columnas de identificación y el día
df_meteo_long = pd.merge(
    df_meteo_melted_values.drop(columns=['Day_raw']),
    df_meteo_melted_validation.drop(columns=['Validation_raw']),
    on=['PROVINCIA', 'MUNICIPIO', 'ESTACION', 'MAGNITUD', 'ANO', 'MES', 'Day'],
    how='left'
)

# Crear la columna FECHA y convertir a datetime, forzando errores a NaT
df_meteo_long['FECHA'] = pd.to_datetime(
    df_meteo_long['ANO'].astype(str) + '-' +
    df_meteo_long['MES'].astype(str) + '-' +
    df_meteo_long['Day'].astype(str),
    errors='coerce'
)

# Eliminar filas con fechas no válidas (NaT)
df_meteo_long.dropna(subset=['FECHA'], inplace=True)

# Convertir VALOR_METEO a numérico, forzando errores a NaN
df_meteo_long['VALOR_METEO'] = pd.to_numeric(df_meteo_long['VALOR_METEO'], errors='coerce')

# Filtrar filas con valores meteorológicos nulos después de la conversión
df_meteo_long.dropna(subset=['VALOR_METEO'], inplace=True)

print("Primeras 5 filas del DataFrame meteorológico en formato largo con la columna FECHA:")
display(df_meteo_long.head())
print(f"Número de filas en df_meteo_long: {len(df_meteo_long)}")

### 4. y 5. Filtrar valores válidos y pivotar `MAGNITUD`

In [ ]:
# Filtrar solo los datos con VALIDACION_METEO 'V' (válido)
df_meteo_valid = df_meteo_long[df_meteo_long['VALIDACION_METEO'] == 'V'].copy()

# Filtrar por las magnitudes específicas que el usuario ha solicitado
parametros_interes = [81, 86, 89]
df_meteo_valid = df_meteo_valid[df_meteo_valid['MAGNITUD'].isin(parametros_interes)]

# Filtrar por la estación 24
df_meteo_valid = df_meteo_valid[df_meteo_valid['ESTACION'] == 24]

# Definir el mapeo de magnitudes a nombres de columnas descriptivos
magnitude_mapping = {
    81: 'velocidad_viento',
    82: 'direccion_viento',
    83: 'temperatura',
    86: 'humedad_relativa',
    87: 'presion_atmosferica',
    88: 'radiacion_solar',
    89: 'precipitaciones'
}

# Pivotar MAGNITUD para que cada una sea una columna separada
df_meteo_pivot = df_meteo_valid.pivot_table(
    index=['PROVINCIA', 'MUNICIPIO', 'ESTACION', 'FECHA'],
    columns='MAGNITUD',
    values='VALOR_METEO',
    aggfunc='mean' # Usar la media si hay múltiples entradas para la misma magnitud/día/estación
).reset_index()

# Renombrar columnas pivotadas
df_meteo_pivot = df_meteo_pivot.rename(columns=magnitude_mapping)

# Rellenar cualquier NaN que pueda surgir por magnitudes no presentes en todos los casos
df_meteo_pivot = df_meteo_pivot.fillna(0) # O un valor adecuado, como la media de la columna

print("Primeras 5 filas del DataFrame meteorológico pivotado:")
display(df_meteo_pivot.head())
print(f"Número de filas en df_meteo_pivot: {len(df_meteo_pivot)}")
print("Columnas y tipos de datos del DataFrame meteorológico pivotado:")
df_meteo_pivot.info()

### 6. Fusionar con `df_merged_final`

In [ ]:
# Fusionar df_merged_final con df_meteo_pivot
df_merged_final_with_meteo = pd.merge(
    df_merged_final,
    df_meteo_pivot,
    on=['FECHA', 'PROVINCIA', 'MUNICIPIO', 'ESTACION'],
    how='left' # Usar left merge para mantener todas las filas de df_merged_final
)

print("Primeras 5 filas de la tabla combinada final con datos meteorológicos:")
display(df_merged_final_with_meteo.head())
print(f"Número de filas en df_merged_final_with_meteo: {len(df_merged_final_with_meteo)}")
print("Columnas y tipos de datos de la tabla combinada final con datos meteorológicos:")
df_merged_final_with_meteo.info()

# Verificar si hay valores nulos en las nuevas columnas meteorológicas
print("\nValores nulos por columna en df_merged_final_with_meteo (especialmente las nuevas):")
print(df_merged_final_with_meteo.isnull().sum())

### Reentrenamiento y Evaluación de Modelos con Características Meteorológicas

In [ ]:
# Definir las características (X) y la variable objetivo (y) con las nuevas variables meteorológicas
X_meteo = df_merged_final_pivot[[
    'intensidad_total_diaria', 'ocupacion_media_diaria', 'carga_media_diaria',
    'mes', 'dia_semana', 'VALOR_DIARIO_DIA_ANTERIOR',
    'velocidad_viento', 'humedad_relativa', 'precipitaciones'
]]
y_meteo = df_merged_final_pivot['VALOR_DIARIO_MAG_8']

# Manejar los valores NaN resultantes de la operación shift y la fusión meteorológica
df_clean_meteo = df_merged_final_pivot.dropna(subset=X_meteo.columns.tolist() + [y_meteo.name])
X_clean_meteo = df_clean_meteo[X_meteo.columns]
y_clean_meteo = df_clean_meteo[y_meteo.name]

# Dividir los datos en conjuntos de entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train_meteo, X_test_meteo, y_train_meteo, y_test_meteo = train_test_split(
    X_clean_meteo, y_clean_meteo, test_size=0.2, random_state=42
)

print(f"Tamaño del conjunto de entrenamiento (con meteo): {len(X_train_meteo)} filas")
print(f"Tamaño del conjunto de prueba (con meteo): {len(X_test_meteo)} filas")

### 1. Modelo de Regresión Lineal con Datos Meteorológicos (Re-entrenado)

In [ ]:
# Inicializar y entrenar el modelo de Regresión Lineal
linear_model_meteo = LinearRegression()
linear_model_meteo.fit(X_train_meteo, y_train_meteo)

# Realizar predicciones en el conjunto de prueba
y_pred_linear_meteo = linear_model_meteo.predict(X_test_meteo)

# Calcular métricas de evaluación
r2_linear_meteo = r2_score(y_test_meteo, y_pred_linear_meteo)
mae_linear_meteo = mean_absolute_error(y_test_meteo, y_pred_linear_meteo)
rmse_linear_meteo = np.sqrt(mean_squared_error(y_test_meteo, y_pred_linear_meteo))

print("--- Regresión Lineal (con datos meteorológicos) ---")
print(f"R2 Score: {r2_linear_meteo:.4f}")
print(f"Mean Absolute Error (MAE): {mae_linear_meteo:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_linear_meteo:.4f}")

### 2. Modelo de Random Forest con Datos Meteorológicos (Re-entrenado)

In [ ]:
# Inicializar y entrenar el modelo de Random Forest Regressor
rf_model_meteo = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_meteo.fit(X_train_meteo, y_train_meteo)

# Realizar predicciones en el conjunto de prueba
y_pred_rf_meteo = rf_model_meteo.predict(X_test_meteo)

# Calcular métricas de evaluación
r2_rf_meteo = r2_score(y_test_meteo, y_pred_rf_meteo)
mae_rf_meteo = mean_absolute_error(y_test_meteo, y_pred_rf_meteo)
rmse_rf_meteo = np.sqrt(mean_squared_error(y_test_meteo, y_pred_rf_meteo))

print("--- Random Forest Regressor (con datos meteorológicos) ---")
print(f"R2 Score: {r2_rf_meteo:.4f}")
print(f"Mean Absolute Error (MAE): {mae_rf_meteo:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_rf_meteo:.4f}")

### Comparación Visual de Predicciones con Datos Meteorológicos (Re-evaluado)

In [ ]:
plt.figure(figsize=(14, 6))

# Subplot para Regresión Lineal
plt.subplot(1, 2, 1) # 1 fila, 2 columnas, primer gráfico
sns.scatterplot(x=y_test_meteo, y=y_pred_linear_meteo, alpha=0.6)
plt.plot([y_test_meteo.min(), y_test_meteo.max()], [y_test_meteo.min(), y_test_meteo.max()], 'r--', lw=2)
plt.title('Regresión Lineal con Meteo: Predicciones vs Reales')
plt.xlabel('Valores Reales de VALOR_DIARIO')
plt.ylabel('Predicciones de VALOR_DIARIO')
plt.grid(True)

# Subplot para Random Forest Regressor
plt.subplot(1, 2, 2) # 1 fila, 2 columnas, segundo gráfico
sns.scatterplot(x=y_test_meteo, y=y_pred_rf_meteo, alpha=0.6, color='green')
plt.plot([y_test_meteo.min(), y_test_meteo.max()], [y_test_meteo.min(), y_test_meteo.max()], 'r--', lw=2)
plt.title('Random Forest con Meteo: Predicciones vs Reales')
plt.xlabel('Valores Reales de VALOR_DIARIO')
plt.ylabel('Predicciones de VALOR_DIARIO')
plt.grid(True)

plt.tight_layout()
plt.show()

### Análisis de Importancia de Características del Modelo Random Forest

Vamos a extraer las importancias de las características de nuestro modelo de Random Forest entrenado (`rf_model_meteo`) para entender cuáles variables son las más influyentes en la predicción de la calidad del aire. Luego visualizaremos estas importancias para facilitar su interpretación.

In [ ]:
# Obtener la importancia de las características del modelo Random Forest
feature_importances = rf_model_meteo.feature_importances_

# Crear un DataFrame para visualizar las importancias
features_df = pd.DataFrame({
    'Feature': X_train_meteo.columns,
    'Importance': feature_importances
})

# Ordenar las características por importancia de forma descendente
features_df = features_df.sort_values(by='Importance', ascending=False)

print("Importancia de las características según Random Forest:")
display(features_df)

# Visualizar la importancia de las características
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=features_df, palette='viridis')
plt.title('Importancia de las Características en Random Forest Regressor')
plt.xlabel('Importancia')
plt.ylabel('Característica')
plt.tight_layout()
plt.show()

### Reentrenamiento y Evaluación de Modelos con Todas las Características Actuales

Vamos a reentrenar y evaluar los modelos de Regresión Lineal y Random Forest utilizando el conjunto completo de características, incluyendo las meteorológicas y las de estacionalidad, sin descartar ninguna.

In [ ]:
# Definir las características (X) y la variable objetivo (y) con todas las variables
X_re_eval = df_merged_final_pivot[[
    'intensidad_total_diaria', 'ocupacion_media_diaria', 'carga_media_diaria',
    'mes', 'dia_semana', 'VALOR_DIARIO_DIA_ANTERIOR',
    'velocidad_viento', 'humedad_relativa', 'precipitaciones'
]]
y_re_eval = df_merged_final_pivot['VALOR_DIARIO_MAG_8']

# Manejar los valores NaN resultantes de la operación shift y la fusión meteorológica
df_clean_re_eval = df_merged_final_pivot.dropna(subset=X_re_eval.columns.tolist() + [y_re_eval.name])
X_clean_re_eval = df_clean_re_eval[X_re_eval.columns]
y_clean_re_eval = df_clean_re_eval[y_re_eval.name]

# Dividir los datos en conjuntos de entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train_re_eval, X_test_re_eval, y_train_re_eval, y_test_re_eval = train_test_split(
    X_clean_re_eval, y_clean_re_eval, test_size=0.2, random_state=42
)

print(f"Tamaño del conjunto de entrenamiento (re-evaluado): {len(X_train_re_eval)} filas")
print(f"Tamaño del conjunto de prueba (re-evaluado): {len(X_test_re_eval)} filas")

### 1. Modelo de Regresión Lineal (Re-entrenado con todas las características)

In [ ]:
# Inicializar y entrenar el modelo de Regresión Lineal
linear_model_re_eval = LinearRegression()
linear_model_re_eval.fit(X_train_re_eval, y_train_re_eval)

# Realizar predicciones en el conjunto de prueba
y_pred_linear_re_eval = linear_model_re_eval.predict(X_test_re_eval)

# Calcular métricas de evaluación
r2_linear_re_eval = r2_score(y_test_re_eval, y_pred_linear_re_eval)
mae_linear_re_eval = mean_absolute_error(y_test_re_eval, y_pred_linear_re_eval)
rmse_linear_re_eval = np.sqrt(mean_squared_error(y_test_re_eval, y_pred_linear_re_eval))

print("--- Regresión Lineal (re-evaluado) ---")
print(f"R2 Score: {r2_linear_re_eval:.4f}")
print(f"Mean Absolute Error (MAE): {mae_linear_re_eval:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_linear_re_eval:.4f}")

### 2. Modelo de Random Forest (Re-entrenado con todas las características)

In [ ]:
# Inicializar y entrenar el modelo de Random Forest Regressor
rf_model_re_eval = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_re_eval.fit(X_train_re_eval, y_train_re_eval)

# Realizar predicciones en el conjunto de prueba
y_pred_rf_re_eval = rf_model_re_eval.predict(X_test_re_eval)

# Calcular métricas de evaluación
r2_rf_re_eval = r2_score(y_test_re_eval, y_pred_rf_re_eval)
mae_rf_re_eval = mean_absolute_error(y_test_re_eval, y_pred_rf_re_eval)
rmse_rf_re_eval = np.sqrt(mean_squared_error(y_test_re_eval, y_pred_rf_re_eval))

print("--- Random Forest Regressor (re-evaluado) ---")
print(f"R2 Score: {r2_rf_re_eval:.4f}")
print(f"Mean Absolute Error (MAE): {mae_rf_re_eval:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_rf_re_eval:.4f}")

### Comparación Visual de Predicciones (Re-evaluado)

In [ ]:
plt.figure(figsize=(14, 6))

# Subplot para Regresión Lineal
plt.subplot(1, 2, 1) # 1 fila, 2 columnas, primer gráfico
sns.scatterplot(x=y_test_re_eval, y=y_pred_linear_re_eval, alpha=0.6)
plt.plot([y_test_re_eval.min(), y_test_re_eval.max()], [y_test_re_eval.min(), y_test_re_eval.max()], 'r--', lw=2)
plt.title('Regresión Lineal (re-evaluado): Predicciones vs Reales')
plt.xlabel('Valores Reales de VALOR_DIARIO')
plt.ylabel('Predicciones de VALOR_DIARIO')
plt.grid(True)

# Subplot para Random Forest Regressor
plt.subplot(1, 2, 2) # 1 fila, 2 columnas, segundo gráfico
sns.scatterplot(x=y_test_re_eval, y=y_pred_rf_re_eval, alpha=0.6, color='green')
plt.plot([y_test_re_eval.min(), y_test_re_eval.max()], [y_test_re_eval.min(), y_test_re_eval.max()], 'r--', lw=2)
plt.title('Random Forest (re-evaluado): Predicciones vs Reales')
plt.xlabel('Valores Reales de VALOR_DIARIO')
plt.ylabel('Predicciones de VALOR_DIARIO')
plt.grid(True)

plt.tight_layout()
plt.show()

### Guardar la tabla combinada final con datos meteorológicos en un archivo CSV

In [ ]:
# Guardar df_merged_final_pivot en un archivo CSV
df_merged_final_pivot.to_csv('df_merged_final_with_meteo.csv', index=False)
print("DataFrame 'df_merged_final_pivot' guardado exitosamente como 'df_merged_final_with_meteo.csv'")

### Verificación de Magnitudes Únicas en `df_calidad_aire`

Antes de transformar `VALOR_DIARIO` de filas a columnas, es importante entender cuántas `MAGNITUDES` diferentes existen en el DataFrame `df_calidad_aire`. Si solo hay una magnitud, entonces la columna `VALOR_DIARIO` ya representa esa magnitud de forma única, y el cambio solicitado podría no ser aplicable en la forma de crear nuevas columnas.

In [ ]:
magnitudes_unicas = df_calidad_aire['MAGNITUD'].unique()
num_magnitudes = df_calidad_aire['MAGNITUD'].nunique()

print(f"Magnitudes únicas en df_calidad_aire: {magnitudes_unicas}")
print(f"Número de magnitudes únicas: {num_magnitudes}")

if num_magnitudes > 1:
    print("Existen múltiples magnitudes. Se procederá a pivotar el DataFrame.")
    # Pivotar df_calidad_aire para que cada MAGNITUD sea una columna de VALOR_DIARIO
    df_calidad_aire_pivot = df_calidad_aire.pivot_table(
        index=['FECHA', 'PROVINCIA', 'MUNICIPIO', 'ESTACION'],
        columns='MAGNITUD',
        values='VALOR_DIARIO',
        aggfunc='mean' # Usar la media si hay múltiples entradas para la misma fecha/estación/magnitud
    ).reset_index()

    # Renombrar las columnas resultantes para mayor claridad
    new_column_names = {col: f'VALOR_DIARIO_MAG_{col}' if isinstance(col, int) else col for col in df_calidad_aire_pivot.columns}
    df_calidad_aire_pivot = df_calidad_aire_pivot.rename(columns=new_column_names)

    print("Primeras 5 filas de df_calidad_aire pivotado:")
    display(df_calidad_aire_pivot.head())
    print("Columnas de df_calidad_aire pivotado:")
    print(df_calidad_aire_pivot.columns)

    # Ahora fusionar con los datos de tráfico y meteo
    # df_trafico_reagregado_filtrado y df_meteo_pivot ya están preparados

    # Primero fusionamos la calidad del aire pivotada con el tráfico
    df_merged_air_traffic = pd.merge(
        df_calidad_aire_pivot,
        df_trafico_reagregado_filtrado,
        on='FECHA',
        how='inner'
    )

    # Luego fusionamos el resultado con los datos meteorológicos
    df_merged_final_pivot = pd.merge(
        df_merged_air_traffic,
        df_meteo_pivot,
        on=['FECHA', 'PROVINCIA', 'MUNICIPIO', 'ESTACION'],
        how='left' # Usar left merge para mantener todas las filas de df_merged_air_traffic
    )

    print("\nPrimeras 5 filas de la tabla final con MAGNITUDES como columnas:")
    display(df_merged_final_pivot.head())
    print(f"Número de filas en df_merged_final_pivot: {len(df_merged_final_pivot)}")
    print("Columnas y tipos de datos de df_merged_final_pivot:")
    df_merged_final_pivot.info()

    # Verificar si hay valores nulos en las nuevas columnas meteorológicas o de VALOR_DIARIO
    print("\nValores nulos por columna en df_merged_final_pivot (especialmente las nuevas):")
    print(df_merged_final_pivot.isnull().sum())

else:
    print("Solo existe una magnitud. La columna 'VALOR_DIARIO' ya contiene los valores para esta magnitud. No es necesario pivotar. Si deseas renombrar la columna, indícalo.")
    # Si solo hay una magnitud, podemos renombrar la columna 'VALOR_DIARIO' para reflejar la magnitud específica
    if len(magnitudes_unicas) == 1:
        df_merged_final_pivot = df_merged_final_with_meteo.rename(columns={'VALOR_DIARIO': f'VALOR_DIARIO_MAG_{magnitudes_unicas[0]}'})

### Verificación de la Alineación de Fechas

Vamos a comprobar que las fechas en `df_merged_final_pivot` (nuestra tabla más completa hasta ahora) son las mismas que las que estaban en `df_merged_final` (la base de `df_merged_final.csv`).

In [ ]:
# Cargar df_merged_final.csv para comparar (o usar df_merged_final si aún está en memoria)
# Para este ejercicio, asumiremos que df_merged_final_pivot es la tabla actual a verificar.

# Obtener las fechas únicas de df_merged_final y df_merged_final_pivot
dates_from_original_merged = set(df_merged_final['FECHA'].dt.date)
dates_from_current_pivot = set(df_merged_final_pivot['FECHA'].dt.date)

# Comprobar si los conjuntos de fechas son idénticos
if dates_from_original_merged == dates_from_current_pivot:
    print("Las fechas en 'df_merged_final_pivot' y 'df_merged_final' son idénticas.")
    print(f"Número de fechas únicas en ambas tablas: {len(dates_from_current_pivot)}")
else:
    print("¡Advertencia! Las fechas no son completamente idénticas. Analizando diferencias...")
    missing_in_pivot = dates_from_original_merged - dates_from_current_pivot
    missing_in_original = dates_from_current_pivot - dates_from_original_merged
    if missing_in_pivot:
        print(f"Fechas presentes en 'df_merged_final' pero no en 'df_merged_final_pivot': {missing_in_pivot}")
    if missing_in_original:
        print(f"Fechas presentes en 'df_merged_final_pivot' pero no en 'df_merged_final': {missing_in_original}")

In [ ]:
print("Primeras 5 filas de df_merged_final_pivot:")
display(df_merged_final_pivot.head())

print("\nÚltimas 5 filas de df_merged_final_pivot:")
display(df_merged_final_pivot.tail())

In [ ]:
print("Primeras 5 filas de df_merged_re_aggregated:")
display(df_merged_re_aggregated.head())

print("\nÚltimas 5 filas de df_merged_re_aggregated:")
display(df_merged_re_aggregated.tail())

In [ ]:
null_values_df = df_merged_re_aggregated[df_merged_re_aggregated.isnull().any(axis=1)]

if not null_values_df.empty:
    print("Filas con valores nulos en df_merged_re_aggregated:")
    display(null_values_df)
else:
    print("No hay valores nulos en el DataFrame df_merged_re_aggregated.")